# AP Lab 10

# 1) Banking System with Deadlock


In [1]:
import threading
import random
import time

# Account balances
accounts = {'A': 1000, 'B': 500}
locks = {'A': threading.Lock(), 'B': threading.Lock()}

def transfer_money(sender, receiver, amount):
    # Acquire locks in consistent order to prevent deadlock
    first, second = sorted([sender, receiver])
    with locks[first]:
        with locks[second]:
            if accounts[sender] >= amount:
                accounts[sender] -= amount
                accounts[receiver] += amount
                print(f"Transfer ${amount}: {sender}->{receiver} | A=${accounts['A']}, B=${accounts['B']}, Total=${accounts['A']+accounts['B']}")

def random_transfer():
    for _ in range(3):
        sender, receiver = random.sample(['A', 'B'], 2)
        amount = random.randint(1, 100)
        transfer_money(sender, receiver, amount)
        time.sleep(0.01)

# Run transfers
threads = [threading.Thread(target=random_transfer) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Final: A=${accounts['A']}, B=${accounts['B']}, Total=${accounts['A']+accounts['B']}")

Transfer $56: A->B | A=$944, B=$556, Total=$1500
Transfer $61: A->B | A=$883, B=$617, Total=$1500
Transfer $64: A->B | A=$819, B=$681, Total=$1500
Transfer $90: A->B | A=$729, B=$771, Total=$1500
Transfer $82: B->A | A=$811, B=$689, Total=$1500
Transfer $100: A->B | A=$711, B=$789, Total=$1500
Transfer $4: B->A | A=$715, B=$785, Total=$1500
Transfer $86: B->A | A=$801, B=$699, Total=$1500
Transfer $13: A->B | A=$788, B=$712, Total=$1500
Transfer $75: B->A | A=$863, B=$637, Total=$1500
Transfer $47: A->B | A=$816, B=$684, Total=$1500
Transfer $91: A->B | A=$725, B=$775, Total=$1500
Transfer $7: B->A | A=$732, B=$768, Total=$1500
Transfer $2: A->B | A=$730, B=$770, Total=$1500
Transfer $23: A->B | A=$707, B=$793, Total=$1500
Final: A=$707, B=$793, Total=$1500


# 2) Text File Word Count with Threads


In [4]:
import threading

# Create sample text file
with open('textfile.txt', 'w') as f:
    f.write("This is a sample text file. " * 100)
    f.write("It contains multiple words for testing. " * 100)
    f.write("We will count words using multiple threads. " * 100)

def count_words(text_chunk, index, results):
    count = len(text_chunk.split())
    results[index] = count
    print(f"Thread {index}: {count} words")

# Read file
with open('textfile.txt', 'r') as f:
    text = f.read()

# Split into 5 parts
chunk_size = len(text) // 5
chunks = [text[i*chunk_size:(i+1)*chunk_size if i < 4 else len(text)] 
          for i in range(5)]

# Count words with threads
results = [0] * 5
threads = [threading.Thread(target=count_words, args=(chunks[i], i, results)) 
           for i in range(5)]

for t in threads: t.start()
for t in threads: t.join()

print(f"Total words: {sum(results)}")

Thread 0: 480 words
Thread 1: 372 words
Thread 2: 336 words
Thread 3: 357 words
Thread 4: 356 words
Total words: 1901


# 3) Matrix Multiplication with Threads


In [3]:
import threading
import time

def multiply_row(A, B, C, row, lock):
    n = len(B[0])
    m = len(B)
    result = [sum(A[row][k] * B[k][j] for k in range(m)) for j in range(n)]
    with lock:
        C[row] = result

# Sequential version
def sequential_multiply(A, B):
    n = len(A)
    m = len(B[0])
    C = [[sum(A[i][k] * B[k][j] for k in range(len(B))) 
          for j in range(m)] for i in range(n)]
    return C

# Threaded version
def threaded_multiply(A, B):
    n = len(A)
    C = [[0] * len(B[0]) for _ in range(n)]
    lock = threading.Lock()
    
    threads = [threading.Thread(target=multiply_row, args=(A, B, C, i, lock)) 
               for i in range(n)]
    for t in threads: t.start()
    for t in threads: t.join()
    return C

# Test with matrices
N = 100
A = [[random.randint(1, 10) for _ in range(N)] for _ in range(N)]
B = [[random.randint(1, 10) for _ in range(N)] for _ in range(N)]

# Compare performance
start = time.time()
C1 = sequential_multiply(A, B)
print(f"Sequential: {time.time() - start:.4f}s")

start = time.time()
C2 = threaded_multiply(A, B)
print(f"Threaded: {time.time() - start:.4f}s")

Sequential: 0.1009s
Threaded: 0.1406s
